# 61 — STAR Bullet Generator
**Goal:** Generate STAR-format resume bullets from minimal input.

Ch. 60 rewrote existing bullets; this chapter generates them from scratch. Given a role, company, duration, and a one-line context, the LLM produces three STAR-compliant bullets — each with a strong action verb, a quantified result, and a different angle (technical, leadership, process). The output parser then turns the model's numbered list into clean bullet strings.

**Why it matters for resumes / ATS:** candidates who "did things" rarely know how to say it — the gap between "Built ML models" and "Developed production NLP models at Google processing 10M+ daily queries, reducing response latency by 35%" is exactly what an ATS scores on. STAR generation closes that gap automatically and consistently, which is also why every generated bullet is *required* to carry a metric.

## 1. STAR Structure

STAR is a four-slot template for a single bullet: **Situation** (context: team size, scope, timeframe), **Task** (what needed doing), **Action** (what *you* did — the verb), **Result** (quantified outcome: %, $, time saved). In practice the four slots compress into one dense sentence — the Situation and Task anchor it, the Action verb leads it, the Result closes it with a number.

**What the code does:** prints the four components and demonstrates the compression on raw input: `"Built ML models at Google"` — a fully-formed bullet that injects `(T)` for the task ("production ML models"), `(S)` for the context ("at Google"), `(A)` for the tooling ("using TensorFlow"), and `(R)` twice for results ("95% accuracy", "10M+ daily predictions"). Note the ordering — the ATS reads action first, so the verb leads even though STAR spells Situation first.

In [ ]:
print('''STAR format:
S - Situation: Context (team size, project scope, timeframe)
T - Task: What needed to be done
A - Action: What YOU did (strong action verb)
R - Result: Quantified outcome (%, $, time saved)

From raw input like:
  "Built ML models at Google"
  → "Developed production ML models (T) at Google (S), achieving 95% accuracy (R) using TensorFlow (A), serving 10M+ daily predictions (R)"''')

## 2. Generator Prompt Template

`STAR_PROMPT` is a `.format()`-able template: four named slots (`role`, `company`, `duration`, `context`) plus four hard requirements — every bullet starts with a strong action verb, every bullet includes a quantified result, STAR is used *naturally within* each bullet (not as labeled sections), and the three bullets cover different aspects: technical achievement, leadership, and process improvement.

**What the code does:**
- The template ends with `Generate exactly 3 bullets:` — the count constraint is explicit because "exactly" is the only way to get a predictable number back.
- `experience` holds the sample record (`Senior Data Scientist`, `Google`, `2020-2023`, NLP-team context) and `STAR_PROMPT.format(**experience)` fills the slots.

**Expected (verified by running):** the printed prompt is the full template with the four fields substituted. **With a key**, the model is expected to return exactly three numbered bullets — one technical, one leadership, one process, each with a metric; the "different aspects" requirement exists to stop the model from generating three near-duplicates.

In [ ]:
STAR_PROMPT = """Generate 3 STAR-format bullet points from the following experience data.

Role: {role}
Company: {company}
Duration: {duration}
Context: {context}

Requirements:
- Each bullet starts with a strong action verb
- Each bullet includes a quantified result
- Use the STAR format naturally within each bullet
- Different aspects: technical achievement, leadership, process improvement

Generate exactly 3 bullets:"""

experience = {
    "role": "Senior Data Scientist",
    "company": "Google",
    "duration": "2020-2023",
    "context": "NLP team building ML pipelines for search"
}
print(STAR_PROMPT.format(**experience))

## 3. Output Parser

Models return numbered lists; the pipeline needs plain bullets. `parse_star_bullets()` filters the response to list-like lines and strips the leading markers — with two caveats worth knowing.

**What the code does:**
- Keeps only non-empty lines that start with a digit, `-`, or `*` — this is what drops the model's preamble ("Here are your bullets:") and trailing commentary.
- Strips leading markers with `re.sub(r"^[\\d\\.\\s\\-\\*]+", "", b)` and drops anything shorter than 20 chars.

**Expected (verified by running):** on the three-bullet sample the parser returns all three lines — *including* the `1. `, `2. `, `3. ` prefixes, because the regex's `\\d` is a doubled backslash in a raw string (a literal backslash, not a digit class), so the numbering is never stripped. The `len(b) > 20` filter hides the damage, but the prefixes still leak into the final bullets; the intended pattern is `r"^[\d.\s\-\*]+"`. Also note the cell uses `re` without importing it — run it in a kernel that already has `re` (e.g. after Ch. 60) or it raises `NameError`.

In [ ]:
def parse_star_bullets(text):
    """Parse generated bullets from LLM output."""
    lines = text.strip().split("\n")
    bullets = [l.strip() for l in lines if l.strip() and (
        l.strip()[0].isdigit() or l.strip().startswith("-") or l.strip().startswith("*")
    )]
    bullets = [re.sub(r"^[\\d\\.\\s\\-\\*]+", "", b) for b in bullets]
    return [b for b in bullets if len(b) > 20]

sample = """1. Developed production NLP models at Google processing 10M+ daily queries, reducing response latency by 35%
2. Led cross-functional team of 5 ML engineers, delivering 3 major product launches on schedule
3. Implemented automated model monitoring pipeline, reducing incident response time from 2 hours to 15 minutes"""

print("Parsed STAR bullets:")
for b in parse_star_bullets(sample):
    print(f"  → {b[:70]}...")

## Summary: STAR generator turns raw experience into quantified, impactful bullets. Always require metrics.

**A metric per bullet — the number is what makes a claim verifiable.**

Given role, company, duration, and context, the generator produces three bullets that each start with an action verb, embed STAR naturally, and carry a quantified result — and the parser turns the model's numbered output into clean strings. Requiring a metric on every bullet is the guardrail: it forces the model to invent (or at least commit to) a number, which is exactly what ATS scoring and human recruiters reward.

This chapter's generation machinery feeds Ch. 62, where the same LLM turns a full resume + JD analysis into role-aware career advice.